# 23 — Partitioning & Consistent Hashing

### Why this exists
Partitioning spreads work and storage, but a bad key can create a hotspot. This lab compares balanced hashing with a skewed workload and a simple ring.

### Learning objectives
hash keys; measure partition distribution; expose hot keys; reason about repartitioning

### Prerequisite
Python hashing and counters

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/23-partitioning-consistent-hashing.svg)

> **Question to carry through the lab:** What happens when one customer or tenant generates 100× more traffic than the others?

## Capability contract

**Capability:** C05 — Storage Consistency & Distributed Systems  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Explain a distributed trade-off


## Mental model
Partitioning is a workload decision. Hashing can spread ordinary keys well while a hot key still concentrates work. Range partitioning helps ordered access but has its own hotspot boundaries.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S08 — Replication and consistency (DATA-1110)** → [`../work_simulations/08_acme_commerce_replica_lag/README.md`](../work_simulations/08_acme_commerce_replica_lag/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
from collections import Counter
keys=[f"tenant-{i}" for i in range(20)] + ["tenant-hot"]*100
counts=Counter(hash(k)%4 for k in keys)
print("hash partition counts",dict(counts))

In [ ]:
hot_key="tenant-hot"
print("hot key routed to",hash(hot_key)%4,"with",keys.count(hot_key),"events")

In [ ]:
# A deterministic hash ring approximation.
def ring_bucket(key,n): return sum(key.encode())%n
print(Counter(ring_bucket(k,4) for k in keys))

In [ ]:
# Break/improve: salt a hot key across virtual buckets, then reconstruct by tenant.
salted=[f"{hot_key}#{i%8}" for i in range(100)]
print("salted spread",Counter(ring_bucket(k,4) for k in salted))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production partitioning requires skew monitoring, key design, rebalancing, locality, query patterns and operational migration plans.

## 9. Independent challenge
Model range partitioning for dates and decide which access pattern should choose range, hash, or a composite strategy. Include the cost of moving data when partitions change.

### Evidence to keep
partition counts, hot-key evidence, salted distribution, and partitioning decision matrix

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.